In [17]:
pip install tensorflow

### Import Datasets

In [18]:
import pandas as pd
import numpy as np
import joblib
import json
import tensorflow as tf
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [19]:
# =============================================================================
# STEP 1: Define FeatureProcessor Class
# =============================================================================

class FeatureProcessor:
    def __init__(self, feature_selection_info, scalers):
        """
        Initialize with saved feature selection information and scalers

        Parameters:
        feature_selection_info: dict containing selected features and scaling info
        scalers: dict containing trained scalers
        """
        self.selected_features = feature_selection_info['selected_features']
        self.robust_features = feature_selection_info['robust_scaling_features']
        self.standard_features = feature_selection_info['standard_scaling_features']
        self.scalers = scalers

    def process_testing_dataset(self, test_df):
        """
        Apply same feature selection and scaling to testing dataset

        Parameters:
        test_df: pandas DataFrame with testing data (contains both benign and attack)

        Returns:
        processed_df: DataFrame with selected features and applied scaling
        labels: Series with attack labels (if 'attack' column exists)
        """
        print(f"Processing testing dataset...")
        print(f"Original dataset shape: {test_df.shape}")

        # Handle missing values
        test_df_clean = test_df.dropna()
        print(f"After removing NaN: {test_df_clean.shape}")

        # Extract labels if attack column exists
        labels = None
        if 'attack' in test_df_clean.columns:
            labels = test_df_clean['attack'].copy()
            print(f"Labels found: {labels.value_counts().to_dict()}")

        # Check if all selected features exist in test dataset
        missing_features = []
        available_features = []

        for feature in self.selected_features:
            if feature in test_df_clean.columns:
                available_features.append(feature)
            else:
                missing_features.append(feature)

        if missing_features:
            print(f"⚠️ Missing features in test dataset: {len(missing_features)}")
            print(f"Missing: {missing_features[:10]}{'...' if len(missing_features) > 10 else ''}")

        print(f"Available features: {len(available_features)}/{len(self.selected_features)}")

        # Extract available features
        processed_df = test_df_clean[available_features].copy()

        # Apply same scaling transformations
        if 'robust' in self.scalers and self.robust_features:
            robust_features_available = [f for f in self.robust_features if f in available_features]
            if robust_features_available:
                processed_df[robust_features_available] = self.scalers['robust'].transform(
                    processed_df[robust_features_available]
                )
                print(f"Applied robust scaling to {len(robust_features_available)} features")

        if 'standard' in self.scalers and self.standard_features:
            standard_features_available = [f for f in self.standard_features if f in available_features]
            if standard_features_available:
                processed_df[standard_features_available] = self.scalers['standard'].transform(
                    processed_df[standard_features_available]
                )
                print(f"Applied standard scaling to {len(standard_features_available)} features")

        print(f"Final processed dataset shape: {processed_df.shape}")
        print("✅ Testing dataset processed successfully")

        return processed_df, labels

In [20]:
# =============================================================================
# STEP 2: Helper Functions
# =============================================================================

def load_feature_processor(save_path="feature_processor.pkl"):
    """Load saved feature processor"""
    processor_data = joblib.load(save_path)

    feature_processor = FeatureProcessor(
        processor_data['feature_selection_info'],
        processor_data['scalers']
    )

    print(f"✅ Feature processor loaded from: {save_path}")
    return feature_processor

def evaluate_processed_data(X_test, y_test):
    """Quick evaluation of processed testing data"""
    print(f"\n📊 PROCESSED TESTING DATA SUMMARY:")
    print(f"Features shape: {X_test.shape}")
    print(f"Feature names: {list(X_test.columns)}")

    if y_test is not None:
        print(f"Labels distribution:")
        print(y_test.value_counts())

        # Convert labels to binary (0=benign, 1=attack)
        y_binary = (y_test != 'benign').astype(int)
        print(f"Binary labels: Benign={np.sum(y_binary==0)}, Attack={np.sum(y_binary==1)}")

        return X_test, y_binary
    else:
        print("No labels found in dataset")
        return X_test, None

In [21]:
# =============================================================================
# STEP 3: Load Feature Processor (FIRST!)
# =============================================================================

print("=== STEP 3: Loading Feature Processor ===")
feature_processor = load_feature_processor("feature_processor.pkl")

=== STEP 3: Loading Feature Processor ===
✅ Feature processor loaded from: feature_processor.pkl


In [22]:
# =============================================================================
# STEP 4: Process Testing Dataset
# =============================================================================

print("\n=== STEP 4: Processing Testing Dataset ===")
from google.colab import drive
drive.mount('/content/drive')
# Load your testing dataset (UPDATE THIS PATH!)
test_csv_path = "/content/drive/MyDrive/FYP/combined_security_dataset.csv"
test_df = pd.read_csv(test_csv_path)
print(f"Loaded testing dataset: {test_csv_path}")

# Process dataset using feature processor
X_test, y_test = feature_processor.process_testing_dataset(test_df)

# Evaluate processed data
X_test_final, y_test_binary = evaluate_processed_data(X_test, y_test)


=== STEP 4: Processing Testing Dataset ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded testing dataset: /content/drive/MyDrive/FYP/combined_security_dataset.csv
Processing testing dataset...
Original dataset shape: (99807, 40)
After removing NaN: (90055, 40)
Labels found: {'benign': 70349, 'attack': 19706}
Available features: 12/12
Applied robust scaling to 11 features
Applied standard scaling to 1 features
Final processed dataset shape: (90055, 12)
✅ Testing dataset processed successfully

📊 PROCESSED TESTING DATA SUMMARY:
Features shape: (90055, 12)
Feature names: ['fwd_header_size_min', 'down_up_ratio', 'bwd_last_window_size', 'fwd_header_size_tot', 'fwd_PSH_flag_count', 'fwd_last_window_size', 'payload_bytes_per_second', 'flow_duration', 'flow_RST_flag_count', 'fwd_data_pkts_tot', 'bwd_pkts_tot', 'fwd_pkts_per_sec']
Labels distribution:
attack
benign    70349
attack    19706
Name: count, dt

In [23]:
# =============================================================================
# STEP 5: Load Trained Models (AFTER processing features!)
# =============================================================================

print("\n=== STEP 5: Loading Trained Models ===")

model_path = "saved_anomaly_models"  # Update this path if needed

# Load sklearn models
ocsvm_model = joblib.load(f"{model_path}/ocsvm_model.pkl")
isolation_forest_model = joblib.load(f"{model_path}/isolation_forest_model.pkl")
lof_model = joblib.load(f"{model_path}/lof_model.pkl")

# Load autoencoder
autoencoder_model = tf.keras.models.load_model(f"{model_path}/autoencoder_model.keras")

# Load autoencoder threshold
with open(f"{model_path}/autoencoder_info.json", 'r') as f:
    autoencoder_info = json.load(f)
autoencoder_threshold = autoencoder_info['threshold']

print("✅ All models loaded successfully!")


=== STEP 5: Loading Trained Models ===
✅ All models loaded successfully!


In [24]:
# =============================================================================
# STEP 6: Test Models on Processed Data
# =============================================================================

print("\n=== STEP 6: Testing Models ===")

def test_processed_dataset(X_test_processed):
    """
    Test processed dataset with all loaded models

    Parameters:
    X_test_processed: preprocessed DataFrame with selected features and scaling applied

    Returns:
    results: dictionary with predictions from each model
    """

    # Convert to numpy array for model input
    X_test_array = X_test_processed.values
    print(f"Testing on {X_test_array.shape[0]} samples with {X_test_array.shape[1]} features")

    results = {}

    # Test OneClassSVM
    ocsvm_pred = ocsvm_model.predict(X_test_array)
    ocsvm_pred = (ocsvm_pred == -1).astype(int)  # Convert to 1=anomaly, 0=normal
    results['OneClassSVM'] = ocsvm_pred
    print(f"OneClassSVM: {np.sum(ocsvm_pred)} anomalies detected")

    # Test Isolation Forest
    if_pred = isolation_forest_model.predict(X_test_array)
    if_pred = (if_pred == -1).astype(int)  # Convert to 1=anomaly, 0=normal
    results['IsolationForest'] = if_pred
    print(f"IsolationForest: {np.sum(if_pred)} anomalies detected")

    # Test LOF
    lof_pred = lof_model.predict(X_test_array)
    lof_pred = (lof_pred == -1).astype(int)  # Convert to 1=anomaly, 0=normal
    results['LOF'] = lof_pred
    print(f"LOF: {np.sum(lof_pred)} anomalies detected")

    # Test Autoencoder
    reconstructed = autoencoder_model.predict(X_test_array)
    mse = np.mean(np.power(X_test_array - reconstructed, 2), axis=1)
    autoencoder_pred = (mse > autoencoder_threshold).astype(int)
    results['Autoencoder'] = autoencoder_pred
    print(f"Autoencoder: {np.sum(autoencoder_pred)} anomalies detected")

    # Summary
    total_samples = len(X_test_array)
    print(f"\n📊 SUMMARY for {total_samples} samples:")
    for model_name, predictions in results.items():
        anomaly_count = np.sum(predictions)
        anomaly_rate = (anomaly_count / total_samples) * 100
        print(f"  {model_name}: {anomaly_count} anomalies ({anomaly_rate:.2f}%)")

    return results

# Run the tests
results = test_processed_dataset(X_test_final)



=== STEP 6: Testing Models ===
Testing on 90055 samples with 12 features
OneClassSVM: 31976 anomalies detected
IsolationForest: 27472 anomalies detected
LOF: 84062 anomalies detected
2815/2815 ━━━━━━━━━━━━━━━━━━━━ 2s 795us/step
Autoencoder: 8027 anomalies detected

📊 SUMMARY for 90055 samples:
  OneClassSVM: 31976 anomalies (35.51%)
  IsolationForest: 27472 anomalies (30.51%)
  LOF: 84062 anomalies (93.35%)
  Autoencoder: 8027 anomalies (8.91%)


In [25]:
# =============================================================================
# STEP 7: Evaluate Performance (if labels available)
# =============================================================================

print("\n=== STEP 7: Performance Evaluation ===")

if y_test_binary is not None:
    print("Evaluating model performance against true labels...")

    for model_name, predictions in results.items():
        print(f"\n{model_name} Performance:")
        print("="*40)

        # Calculate metrics
        accuracy = accuracy_score(y_test_binary, predictions)
        print(f"Accuracy: {accuracy:.4f}")

        # Classification report
        print("\nClassification Report:")
        print(classification_report(y_test_binary, predictions,
                                  target_names=['Benign', 'Attack']))

        # Confusion matrix
        cm = confusion_matrix(y_test_binary, predictions)
        print(f"Confusion Matrix:")
        print(f"True Negatives (Benign correctly identified): {cm[0,0]}")
        print(f"False Positives (Benign labeled as Attack): {cm[0,1]}")
        print(f"False Negatives (Attack labeled as Benign): {cm[1,0]}")
        print(f"True Positives (Attack correctly identified): {cm[1,1]}")

else:
    print("No labels available - cannot evaluate performance")


=== STEP 7: Performance Evaluation ===
Evaluating model performance against true labels...

OneClassSVM Performance:
Accuracy: 0.5730

Classification Report:
              precision    recall  f1-score   support

      Benign       0.77      0.64      0.70     70349
      Attack       0.21      0.34      0.26     19706

    accuracy                           0.57     90055
   macro avg       0.49      0.49      0.48     90055
weighted avg       0.65      0.57      0.60     90055

Confusion Matrix:
True Negatives (Benign correctly identified): 44989
False Positives (Benign labeled as Attack): 25360
False Negatives (Attack labeled as Benign): 13090
True Positives (Attack correctly identified): 6616

IsolationForest Performance:
Accuracy: 0.5442

Classification Report:
              precision    recall  f1-score   support

      Benign       0.73      0.65      0.69     70349
      Attack       0.11      0.16      0.13     19706

    accuracy                           0.54     90055
   m

In [26]:
# =============================================================================
# STEP 8: Save Results
# =============================================================================

print("\n=== STEP 8: Saving Results ===")

# Save results
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

# Save predictions
df_results = pd.DataFrame(results)
results_file = f"model_predictions_{timestamp}.csv"
df_results.to_csv(results_file, index=False)
print(f"✅ Model predictions saved to: {results_file}")

# Save processed features
features_file = f"processed_test_features_{timestamp}.csv"
X_test_final.to_csv(features_file, index=False)
print(f"✅ Processed features saved to: {features_file}")

# Save labels if available
if y_test_binary is not None:
    labels_file = f"test_labels_{timestamp}.csv"
    pd.DataFrame({'true_label': y_test_binary}).to_csv(labels_file, index=False)
    print(f"✅ True labels saved to: {labels_file}")

print(f"\n🎉 Testing complete! All results saved with timestamp: {timestamp}")


=== STEP 8: Saving Results ===
✅ Model predictions saved to: model_predictions_20250714_110602.csv
✅ Processed features saved to: processed_test_features_20250714_110602.csv
✅ True labels saved to: test_labels_20250714_110602.csv

🎉 Testing complete! All results saved with timestamp: 20250714_110602
